In [ ]:
import os
import json
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
compile_pgf = False
if compile_pgf:
	matplotlib.use("pgf")
	matplotlib.rcParams.update({
	    "pgf.texsystem": "pdflatex",
	    'font.family': 'serif',
	    'text.usetex': True,
	    "axes.formatter.use_mathtext": True
	})

In [ ]:
filename = "completeResults_sequentialDelays_2026-06-05"
filename = "replan_@MAEDeR_maze-128-128-1-even-1-k50_2026-07-29-16-03_f8_oTrue_seed42_25delays_mINF"
folder = "output/maze1"

filename = "replan_@MAEDeR_maze-128-128-1-even-1-k50_2026-07-30-09-28_f0_oTrue_seed42_25delays_mINF"
folder = "db_output"


if "FlexSIPP" in filename:
    raise ValueError("Should enter the @MAEDeR file not the FlexSIPP file")
filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", folder, f"{filename}.json")
complete_result = {
    "@MAEDeR": json.load(open(filepath, "r")),
    "FlexSIPP": json.load(open(filepath.replace("@MAEDeR", "FlexSIPP"), "r"))
}


## Process

In [ ]:
df = pd.DataFrame(columns=["Delay Idx", "Delay Agent", "Delayed Starttime", "Delay Amount", "Cumulative Delay", "Total FlexSIPP" ,"Total @MAEDeR", "Fail FlexSIPP", "Fail @MAEDeR"])
rows = 0
paths = {
    "FlexSIPP": {a: [r["arrival"][1]]  for a, r in complete_result["@MAEDeR"]["delay0"]["initial_paths"].items()},
    "@MAEDeR": {a: [r["arrival"][1]]  for a, r in complete_result["@MAEDeR"]["delay0"]["initial_paths"].items()}
}
delays = ["" for a in complete_result["@MAEDeR"]["delay0"]["initial_paths"]]
cumulative_delay = 0
for delay in complete_result["FlexSIPP"]:
    if complete_result["FlexSIPP"][delay]:
        for a, r in complete_result["FlexSIPP"][delay]["arrival_times"].items():
            paths["FlexSIPP"][a].append(r["arrival"][1])
        for a, r in complete_result["@MAEDeR"][delay]["arrival_times"].items():
            paths["@MAEDeR"][a].append(r["arrival"][1])
        delays[int(complete_result["FlexSIPP"][delay]["delay_agent"])-1] = str(complete_result["FlexSIPP"][delay]["delay_agent"])
        assert complete_result["FlexSIPP"][delay]["delay_agent"] == complete_result["@MAEDeR"][delay]["delay_agent"]
        cumulative_delay += (complete_result["FlexSIPP"][delay]["delayed_start_time"] - complete_result["FlexSIPP"][delay]["original_start_time"])
        print(f"Delay {delay} has start time {complete_result['FlexSIPP'][delay]['original_start_time']} and delayed start {complete_result['FlexSIPP'][delay]['delayed_start_time']}, cumulative delay={cumulative_delay}")
        df.loc[rows] = [
			delay,
			complete_result["FlexSIPP"][delay]["delay_agent"],
			complete_result["FlexSIPP"][delay]["delayed_start_time"],
			complete_result["FlexSIPP"][delay]["delayed_start_time"] - complete_result["FlexSIPP"][delay]["original_start_time"],
			cumulative_delay,
			sum([(paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0]) for a in paths["FlexSIPP"]]),
			sum([(paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0]) for a in paths["@MAEDeR"]]),
			not (complete_result["FlexSIPP"][delay] and complete_result["FlexSIPP"][delay]["unique_routes_safe"]),
			not (complete_result["@MAEDeR"][delay] and complete_result["@MAEDeR"][delay]["unique_routes_safe"]),
		]
        rows += 1

In [ ]:
# json.dump(complete_result, open("completeResults_sequentialDelays_2026-06-25.json", "w"), indent=4)

In [ ]:
print("@MAEDeR failed", len(df[df["Fail @MAEDeR"]]), "times and FlexSIPP failed", len(df[df["Fail FlexSIPP"]]), "times")

#### Print individual differences per agent MAEDeR and FlexSIPP

In [ ]:
print("\t\t@MAEDEr FLexSIPP")
for a in paths["FlexSIPP"]:
    print("Agent", a, "\t", paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0], "\t", paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0], "\t",
          (paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0]) / (paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0]) * 100 if paths["@MAEDeR"][a][-1] -  paths["@MAEDeR"][a][0] != 0 else '100', "%")
print("Agent", a, "\t", 
    sum([(paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0]) for a in paths["FlexSIPP"]]),"\t", 
    sum([(paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0]) for a in paths["@MAEDeR"]]),
)

## Show cumulative delay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))


colors = [
    (0.83527, 0.886029, 0.102646),
    (0.283187, 0.125848, 0.44496),
    (0.132268, 0.655014, 0.519661),
]
print("Max @MAEDeR", df['Total @MAEDeR'].max(), "Max FlexSIPP", df['Total FlexSIPP'].max(), "FlexSIPP / @MAEDeR = ", round(df['Total FlexSIPP'].max() / df['Total @MAEDeR'].max() * 100, 2), "%")

df.plot(ax=ax, x="Delay Idx", y="Cumulative Delay", label="Input Delay", color=colors[0])
df.plot(ax=ax, x="Delay Idx", y="Total FlexSIPP", label="Total Delay FlexSIPP", color=colors[1])
df.plot(ax=ax, x="Delay Idx", y="Total @MAEDeR", label="Total Delay @MAEDeR", color=colors[2])


ax.scatter(df.index[df['Fail FlexSIPP']], df.loc[df['Fail FlexSIPP'], 'Total FlexSIPP'], marker='x', color=colors[1], zorder=5, linewidth=2)
ax.scatter(df.index[df['Fail @MAEDeR']], df.loc[df['Fail @MAEDeR'], 'Total @MAEDeR'], marker='x', color=colors[2], zorder=5, linewidth=2)

ticks = [int(x.replace("delay", "")) + 1 if i % 2 == 0 else "" for i, x in enumerate(df["Delay Idx"].unique())]

fonts = 16

ax.set_xticks(range(len(df["Delay Idx"].unique())))
ax.set_xticklabels(ticks, fontsize=fonts)
ax.set_ylabel("Total Delay", fontsize=fonts)
ax.set_xlabel("Delay Index", fontsize=fonts)
ax.legend(fontsize=fonts)
ax.set_xlabel(ax.get_xlabel(), fontsize=fonts)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=fonts)
filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "sequential_delay_updates")
plt.tight_layout()
extension = "pgf" if compile_pgf else "png"
plt.savefig(f"{filepath}.{extension}", dpi=600)
plt.show()
plt.close()
print("Max @MAEDeR", df['Total @MAEDeR'].max(), "Max FlexSIPP", df['Total FlexSIPP'].max(), "FlexSIPP / @MAEDeR = ", round(df['Total FlexSIPP'].max() / df['Total @MAEDeR'].max() * 100, 2), "%")

### Plot the difference per agent

In [ ]:
show_individual_diffs = False
if show_individual_diffs:
	fig, ax = plt.subplots(2,1, figsize=(8, 8))

	for num, alg in enumerate(paths):
		agents = []
		for i, (x, ys) in enumerate(paths[alg].items()):
			y_start = ys[0]
			y_end = ys[-1]
			y_min = min(ys)
			y_max = max(ys)

			# Draw the full range as a thin background line
			agents.append(x)
			ax[num].plot([i, i], [y_min, y_max], color="lightgray", linewidth=4, zorder=1)

			color = "gray"
			if str(x) in delays:
				color = "red"

			# Mark intermediate points
			for y in ys[1:-1]:
				ax[num].scatter(i, y, color=color, s=30, zorder=3)

			# Draw arrow from first to last value
			ax[num].annotate(
				"",
				xy=(i, y_end),
				xytext=(i, y_start),
				arrowprops=dict(arrowstyle="->", color="black", lw=2),
				zorder=2,
			)


		ax[num].set_xticks([int(x)-1 for x in agents])
		ax[num].set_xticklabels([str(x)  if i % 4 == 0 else "" for i,x in enumerate(agents)], fontsize=12)
		ax[num].set_xlabel("Agent", fontsize=12)
		ax[num].set_ylabel("Arrival Time", fontsize=12)
		ax[num].set_yticklabels(ax[num].get_yticklabels(), fontsize=12)
		ax[num].set_title(alg)
	plt.tight_layout()
	plt.savefig(f"{filepath}.{extension}", dpi=600)
	plt.show()